In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, UpSampling2D
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Set paths
image_folder = os.path.join('dataset', 'images')
mask_folder = os.path.join('dataset', 'masks')

# Parameters
img_size = (256, 256)

# Load and resize images
def load_images_from_folder(folder, color_mode='rgb'):
    images = []
    for filename in sorted(os.listdir(folder)):
        img = load_img(os.path.join(folder, filename), target_size=img_size, color_mode=color_mode)
        img = img_to_array(img) / 255.0
        images.append(img)
    return np.array(images)

# Load images and masks
X = load_images_from_folder(image_folder, color_mode='rgb')
y = load_images_from_folder(mask_folder, color_mode='grayscale')

# Threshold masks to binary
y = (y > 0.5).astype(np.float32)

print("Images shape:", X.shape)
print("Masks shape:", y.shape)

# Split into train and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(256, 256, 3)),
    MaxPooling2D(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    UpSampling2D(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    UpSampling2D(),
    Conv2D(1, (1, 1), activation='sigmoid', padding='same')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=50, batch_size=2)

# Save model
model.save('segmentation_model.h5')

# Plot training and validation loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()
